##  Text to Vectors and Semantic Similarity

An **embedding** is a dense vector (ordered list of numbers) produced by a model such that texts with **similar meaning** map to **nearby** points in vector space.

**How Text Becomes a Vector:**
1. Raw text
2. Tokenization
3. Encoding (embedding model)
4. Store or compare

Let's preview how a sentence is converted into an embedding using `SentenceTransformer`.

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # Same model used in the Chroma lab below


sentence_a = "Refunds are processed within 5 to 7 business days"  # FAQ-style stored text
sentence_b = "When will I get my money back after returning the item?"  # User-style paraphrase

vectora=model.encode(sentence_a,convert_to_numpy=True)
vectorb=model.encode(sentence_b,convert_to_numpy=True)

print("The length of vectora : ",len(vectora))
print("First five vectors : ",vectora[:5])



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5848.77it/s]


The length of vectora :  384
First five vectors :  [-0.06581643 -0.00359726  0.08447061  0.01167005  0.04429862]


## 3. Prepare Sample Data

Good retrieval starts with clean, small chunks — one idea per row.
Our example is an **e-commerce support knowledge base** with five short FAQs.

In [8]:
records = [  # Each dict = one Chroma row (id, text, metadata)
    {"id": "doc1", "text": "Customers can return products within 30 days of delivery.", "metadata": {"category": "returns"}},
    {"id": "doc2", "text": "Refunds are processed within 5 to 7 business days after the return is approved.", "metadata": {"category": "returns"}},
    {"id": "doc3", "text": "Orders above 499 rupees qualify for free shipping.", "metadata": {"category": "shipping"}},
    {"id": "doc4", "text": "You can reset your password from the account settings page.", "metadata": {"category": "account"}},
    {"id": "doc5", "text": "Express delivery orders usually arrive within 24 to 48 hours.", "metadata": {"category": "shipping"}},
]

# Note: Each dict is one record — like one SQL row.

## 4. Create the Chroma Client and Collection
We will create a Persistent Client (saves to disk) and a new Collection (like an SQL Table) to store our embeddings.

In [ ]:
# creating the client and the collection 
import chromadb
from pprint import pprint

client=chromadb.PersistentClient(path="./chroma_store")

collection=client.get_or_create_collection(
    name="Support_knowledge_base",
    embedding_function=None # We will pass embeddings manually

)

print("Collection : ",collection.name)
print('cont before usert : ',collection.count())

Collection :  Support_knowledge_base
cont before usert :  0


## 5. Add Data to Your Chroma Collection
We must generate vectors for our documents using the exact same `SentenceTransformer` model before writing them to the database.

In [22]:
model=SentenceTransformer("all-MiniLM-L6-v2")

# separating the lists form collection

documents=[]
ids=[]
metadata=[]
for i in records:
    documents.append(i["text"])
    ids.append(i["id"])
    metadata.append(i["metadata"])
# encoding the documents
document_embedding=model.encode(
    documents,
    convert_to_numpy=True,

).tolist()

# upserting the parallel lines 
collection.upsert(
        ids=ids,
    documents=documents,
    metadatas=metadata,
    embeddings=document_embedding,
)
print("Upsert complete ,the total rows are : ",collection.count())# should be 5


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8133.09it/s]


Upsert complete ,the total rows are :  5


## 6. Verify What You Stored
Never demo a search without confirming the *add* operation worked. Let's peek into the DB.

In [24]:
print("Total records : ",collection.count())
pprint(collection.peek())

Total records :  5
{'data': None,
 'documents': ['Customers can return products within 30 days of delivery.',
               'Refunds are processed within 5 to 7 business days after the '
               'return is approved.',
               'Orders above 499 rupees qualify for free shipping.',
               'You can reset your password from the account settings page.',
               'Express delivery orders usually arrive within 24 to 48 hours.'],
 'embeddings': array([[-0.04682997, -0.02426491,  0.0411784 , ..., -0.03994693,
         0.05715275, -0.02282485],
       [-0.06452115, -0.01745071,  0.08278479, ..., -0.04997708,
        -0.02753523, -0.01114142],
       [-0.03890188, -0.04008841,  0.04764163, ..., -0.0512668 ,
         0.02335656, -0.04045212],
       [-0.02485109, -0.05870529, -0.01641148, ...,  0.07588056,
        -0.01731195, -0.07811966],
       [ 0.01435259, -0.03327222,  0.05274634, ..., -0.00148862,
         0.06538489, -0.04434861]], shape=(5, 384)),
 'ids': ['doc

## 7. Retrieve Data with Similarity Search

**Activity: Predict the Top Match**
If a user asks: *"I want to return my shoes and get my money back"*
Which of our 5 FAQs do you think will be the closest mathematical match?

In [26]:
# --- Demo 1: The Returns Query ---
user_query_1 = "I want to return my shoes and get my money back"

query_embedding_1 = model.encode(
    [user_query_1], 
    convert_to_numpy=True
).tolist()  # MUST be same model as document ingest

results_1=collection.query(
    query_embeddings=query_embedding_1,
    n_results=1
)
print(results_1)
print("Query:", user_query_1)
print("\nTop matches:")
for i in range(len(results_1["ids"][0])):
    print(f"Rank {i + 1}")
    print(" ID:", results_1["ids"][0][i])
    print(" Document:", results_1["documents"][0][i])
    print(" Distance:", results_1["distances"][0][i])
    print()

{'ids': [['doc2']], 'embeddings': None, 'documents': [['Refunds are processed within 5 to 7 business days after the return is approved.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'category': 'returns'}]], 'distances': [[1.1705917119979858]]}
Query: I want to return my shoes and get my money back

Top matches:
Rank 1
 ID: doc2
 Document: Refunds are processed within 5 to 7 business days after the return is approved.
 Distance: 1.1705917119979858

